<a href="https://colab.research.google.com/github/SanthiyaElumalai/Algotrade/blob/main/AngelOne.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/angel-one/smartapi-python.git
!pip install -r smartapi-python/requirements_dev.txt
!pip install pyotp
!pip install smartapi-python
!pip install logzero
!pip install websocket-client

fatal: destination path 'smartapi-python' already exists and is not an empty directory.


In [ ]:
import pandas as pd
from SmartApi import SmartConnect
import pyotp
import pytz
from datetime import datetime, timedelta
import time

class AngelLivePaperTrader:
    def __init__(self, client_id, pwd, totp, access_token):
        self.current_totp = pyotp.TOTP(totp).now()
        self.obj = SmartConnect(api_key=access_token)
        self.data = self.obj.generateSession(client_id, pwd, self.current_totp)
        self.exchange = "NSE"
        self.symboltoken = "99926000"
        self.interval = "ONE_MINUTE"
        self.ist = pytz.timezone('Asia/Kolkata')

        # FIX 1: Initializing missing tracking arrays required by the pipeline
        self.formed_resistance_rays = []
        self.active_resistance_breakouts = []
        self.formed_support_rays = []
        self.active_support_breakdowns = []

    def getting_data(self):
        now_time = datetime.now(self.ist) - timedelta(minutes=1)
        end_date = now_time.strftime("%Y-%m-%d %H:%M")
        start_date = (now_time - pd.Timedelta(days=15)).strftime("%Y-%m-%d %H:%M")

        if not self.data or not self.data.get("status"):
            return None

        try:
            historic_params = {
                "exchange": self.exchange,
                "symboltoken": self.symboltoken,
                "interval": self.interval,
                "fromdate": start_date,
                "todate": end_date
            }
            raw_candles = self.obj.getCandleData(historic_params)
            if raw_candles and raw_candles.get("status") and raw_candles.get("data"):
                columns = ["Timestamp", "Open", "High", "Low", "Close", "Volume"]
                dataframe = pd.DataFrame(raw_candles["data"], columns=columns)
                return dataframe
            return None
        except Exception:
            return None

    def get_line_projection(self, start_row, start_val, end_row, end_val, current_row):
        if end_row == start_row:
            return end_val, 0.0
        slope = (end_val - start_val) / (end_row - start_row)
        projected_val = end_val + slope * (current_row - end_row)
        return projected_val, slope

    def getting_logic(self):
        while True:
            try:
                df = self.getting_data()
                if df is None or df.empty or len(df) < 140:
                    time.sleep(10)
                    continue

                df['Timestamp'] = pd.to_datetime(df['Timestamp'])
                df.set_index('Timestamp', inplace=True)
                df.index = df.index.tz_convert(self.ist)
                df = df.dropna(subset=['High', 'Low', 'Close'])

                current_idx = len(df) - 1
                current_close = df['Close'].iloc[-1]

                # 1. MACRO CHANNELS
                macro_block_1 = df.iloc[-60:]
                macro_block_2 = df.iloc[-120:-60]

                macro_c_high_idx, macro_c_low_idx = macro_block_1['High'].iloc[::-1].idxmax(), macro_block_1['Low'].iloc[::-1].idxmin()
                macro_p_high_idx, macro_p_low_idx = macro_block_2['High'].iloc[::-1].idxmax(), macro_block_2['Low'].iloc[::-1].idxmin()

                m_p_high_row = df.index.get_loc(macro_p_high_idx)
                m_c_high_row = df.index.get_loc(macro_c_high_idx)
                macro_c_close_at_high = macro_block_1.loc[macro_c_high_idx, 'Low']
                m_p_low_row  = df.index.get_loc(macro_p_low_idx)
                m_c_low_row  = df.index.get_loc(macro_c_low_idx)
                macro_c_close_at_low = macro_block_1.loc[macro_c_low_idx, 'High']

                macro_rp, _ = self.get_line_projection(m_p_high_row, macro_block_2['High'].max(), m_c_high_row, macro_c_close_at_high, current_idx)
                macro_sp, _ = self.get_line_projection(m_p_low_row,  macro_block_2['Low'].min(),  m_c_low_row,  macro_c_close_at_low,  current_idx)

                # 2. MICRO TRENDLINES
                micro_block_1 = df.iloc[-30:]
                micro_block_2 = df.iloc[-60:-30]

                micro_c_high_idx, micro_c_low_idx = micro_block_1['High'].iloc[::-1].idxmax(), micro_block_1['Low'].iloc[::-1].idxmin()
                micro_p_high_idx, micro_p_low_idx = micro_block_2['High'].iloc[::-1].idxmax(), micro_block_2['Low'].iloc[::-1].idxmin()

                mi_p_high_row = df.index.get_loc(micro_p_high_idx)
                mi_c_high_row = df.index.get_loc(micro_c_high_idx)
                micro_c_close_at_high = micro_block_1.loc[micro_c_high_idx, 'Low']

                mi_p_low_row  = df.index.get_loc(micro_p_low_idx)
                mi_c_low_row  = df.index.get_loc(micro_c_low_idx)
                micro_c_close_at_low = micro_block_1.loc[micro_c_low_idx, 'High']

                micro_rp, micro_slope_r = self.get_line_projection(mi_p_high_row, micro_block_2['High'].max(), mi_c_high_row, micro_c_close_at_high, current_idx)
                micro_sp, micro_slope_s = self.get_line_projection(mi_p_low_row,  micro_block_2['Low'].min(),  mi_c_low_row,  micro_c_close_at_low,  current_idx)

                india_time = datetime.now(self.ist)
                print("macro resistance:", macro_p_high_idx, macro_block_2['High'].iloc[::-1].max(), '\t', macro_c_high_idx, macro_c_close_at_high)
                print("macro support:", macro_p_low_idx, macro_block_2['Low'].iloc[::-1].min(), '\t', macro_c_low_idx, macro_c_close_at_low,'\n')
                print("micro resistance:", micro_p_high_idx, micro_block_2['High'].iloc[::-1].max(), '\t', micro_c_high_idx, micro_c_close_at_high)
                print("micro support:", micro_p_low_idx, micro_block_2['Low'].iloc[::-1].min(), '\t', micro_c_low_idx, micro_c_close_at_low)
                print(f"\n[{india_time.strftime('%H:%M:%S')}] Close: {current_close} | Macro [S:{macro_sp:.1f} R:{macro_rp:.1f}] | Micro [S:{micro_sp:.1f} R:{micro_rp:.1f}]")
                # ===========================================

                # 3. ADVANCED STATE MACHINE PIPELINE
                if macro_sp <= current_close <= macro_rp:

                    # --- RESISTANCE PIPELINE ---
                    recent_peak_high = micro_c_close_at_high

                    if recent_peak_high > current_close and micro_rp > current_close:
                        is_ray_clean = True
                        candles_since_anchor = current_idx - mi_c_high_row

                        for step in range(1, candles_since_anchor + 1):
                            check_idx = mi_c_high_row + step
                            past_ray_val = recent_peak_high + micro_slope_r * step
                            past_high = df['Close'].iloc[check_idx]

                            if past_high > past_ray_val:
                                is_ray_clean = False
                                break

                        if is_ray_clean:
                            is_new_r = not any(r['anchor_time'] == micro_c_high_idx for r in self.formed_resistance_rays) and \
                                       not any(ar['anchor_time'] == micro_c_high_idx for ar in self.active_resistance_breakouts)
                            if is_new_r:
                                self.formed_resistance_rays.append({
                                    'base_val': recent_peak_high,
                                    'slope': micro_slope_r,
                                    'anchor_row': mi_c_high_row,
                                    'anchor_time': micro_c_high_idx,
                                    'last_calculated_val': micro_rp
                                })
                                # SILENT PRINTING: Only displays upon validation
                                print(f"[{india_time.strftime('%H:%M:%S')}] -> [VALID OVERHEAD RESISTANCE] Clean ray stored at {micro_rp:.2f}")

                    # --- SUPPORT PIPELINE ---
                    recent_trough_low = micro_c_close_at_low

                    if recent_trough_low < current_close and micro_sp < current_close:
                        is_ray_clean = True
                        candles_since_anchor = current_idx - mi_c_low_row

                        for step in range(1, candles_since_anchor + 1):
                            check_idx = mi_c_low_row + step
                            past_ray_val = recent_trough_low + micro_slope_s * step
                            past_low = df['Close'].iloc[check_idx]

                            if past_low < past_ray_val:
                                is_ray_clean = False
                                break

                        if is_ray_clean:
                            is_new_s = not any(s['anchor_time'] == micro_c_low_idx for s in self.formed_support_rays) and \
                                       not any(as_['anchor_time'] == micro_c_low_idx for as_ in self.active_support_breakdowns)
                            if is_new_s:
                                self.formed_support_rays.append({
                                    'base_val': recent_trough_low,
                                    'slope': micro_slope_s,
                                    'anchor_row': mi_c_low_row,
                                    'anchor_time': micro_c_low_idx,
                                    'last_calculated_val': micro_sp
                                })
                                # SILENT PRINTING: Only displays upon validation
                                print(f"[{india_time.strftime('%H:%M:%S')}] -> [VALID UNDERLYING SUPPORT] Clean ray stored at {micro_sp:.2f}")

                time.sleep(30)

            except Exception:
                time.sleep(30)

if __name__ == "__main__":
    API_KEY = "cQtdXuXL"
    CLIENT_CODE = "AABZ498970"
    PASSWORD = '1107'
    TOTP_TOKEN = "CGAODQ6O3SGLIBQWUSWFABIL2Y"
    bot = AngelLivePaperTrader(client_id=CLIENT_CODE, pwd=PASSWORD, totp=TOTP_TOKEN, access_token=API_KEY)
    bot.getting_logic()

In [ ]:
import pandas as pd
from SmartApi import SmartConnect
import pyotp
import pytz
from datetime import datetime, timedelta
import time

class AngelLivePaperTrader:
    def __init__(self, client_id, pwd, totp, access_token):
        self.current_totp = pyotp.TOTP(totp).now()
        self.obj = SmartConnect(api_key=access_token)
        self.data = self.obj.generateSession(client_id, pwd, self.current_totp)
        self.exchange = "NSE"
        self.symboltoken = "99926000"
        self.interval = "ONE_MINUTE"
        self.ist = pytz.timezone('Asia/Kolkata')

        # Trackers required by the pipeline
        self.formed_resistance_rays = []
        self.active_resistance_breakouts = []
        self.formed_support_rays = []
        self.active_support_breakdowns = []

    def getting_data(self):
        now_time = datetime.now(self.ist) - timedelta(minutes=1)
        end_date = now_time.strftime("%Y-%m-%d %H:%M")
        start_date = (now_time - pd.Timedelta(days=15)).strftime("%Y-%m-%d %H:%M")

        if not self.data or not self.data.get("status"):
            return None

        try:
            historic_params = {
                "exchange": self.exchange,
                "symboltoken": self.symboltoken,
                "interval": self.interval,
                "fromdate": start_date,
                "todate": end_date
            }
            raw_candles = self.obj.getCandleData(historic_params)
            if raw_candles and raw_candles.get("status") and raw_candles.get("data"):
                columns = ["Timestamp", "Open", "High", "Low", "Close", "Volume"]
                dataframe = pd.DataFrame(raw_candles["data"], columns=columns)
                return dataframe
            return None
        except Exception:
            return None

    def get_line_projection(self, start_row, start_val, end_row, end_val, current_row):
        if end_row == start_row:
            return end_val, 0.0
        slope = (end_val - start_val) / (end_row - start_row)
        projected_val = end_val + slope * (current_row - end_row)
        return projected_val, slope

    def separate_candles(self, df):
        """
        Separates the dataframe into green and red candles,
        returning the most recent candle data for both types.
        """
        green_candles = df[df['Close'] > df['Open']]
        red_candles = df[df['Close'] < df['Open']]

        latest_green = green_candles.iloc[-1] if not green_candles.empty else None
        latest_red = red_candles.iloc[-1] if not red_candles.empty else None

        return latest_green, latest_red, green_candles, red_candles

    def getting_logic(self):
        while True:
            try:
                df = self.getting_data()
                if df is None or df.empty or len(df) < 140:
                    time.sleep(10)
                    continue

                df['Timestamp'] = pd.to_datetime(df['Timestamp'])
                df.set_index('Timestamp', inplace=True)
                df.index = df.index.tz_convert(self.ist)
                df = df.dropna(subset=['Open', 'High', 'Low', 'Close'])

                current_idx = len(df) - 1
                current_close = df['Close'].iloc[-1]

                # Separate candles while retaining complete context
                latest_green, latest_red, green_candles, red_candles = self.separate_candles(df)

                if len(green_candles) < 120 or len(red_candles) < 120:
                    print("Not enough specific color candles to compute levels yet. Waiting...")
                    time.sleep(10)
                    continue

                # Helper function to map candle timestamps back to the primary full 'df' axis
                def get_df_idx(timestamp):
                    return df.index.get_loc(timestamp)

                # -----------------------------------------------------------
                # 1. GREEN CANDLE LOGIC (MACRO & MICRO)
                # -----------------------------------------------------------
                # Macro Channels (Green)
                g_macro_block_1 = green_candles.iloc[-15:]
                g_macro_block_2 = green_candles.iloc[-30:-15]

                g_macro_c_high_idx = g_macro_block_1['High'].iloc[::-1].idxmax()
                g_macro_c_low_idx  = g_macro_block_1['Low'].iloc[::-1].idxmin()
                g_macro_p_high_idx = g_macro_block_2['High'].iloc[::-1].idxmax()
                g_macro_p_low_idx  = g_macro_block_2['Low'].iloc[::-1].idxmin()

                # Get row index positions relative to the full df timeline
                g_m_p_high_row = get_df_idx(g_macro_p_high_idx)
                g_m_c_high_row = get_df_idx(g_macro_c_high_idx)
                g_m_p_low_row  = get_df_idx(g_macro_p_low_idx)
                g_m_c_low_row  = get_df_idx(g_macro_c_low_idx)

                g_macro_p_high_val = g_macro_block_2.loc[g_macro_p_high_idx, 'High']
                g_macro_c_high_val = g_macro_block_1.loc[g_macro_c_high_idx, 'High']
                g_macro_p_low_val  = g_macro_block_2.loc[g_macro_p_low_idx, 'Low']
                g_macro_c_low_val  = g_macro_block_1.loc[g_macro_c_low_idx, 'Low']

                g_macro_rp, _ = self.get_line_projection(g_m_p_high_row, g_macro_p_high_val, g_m_c_high_row, g_macro_c_high_val, current_idx)
                g_macro_sp, _ = self.get_line_projection(g_m_p_low_row,  g_macro_p_low_val,  g_m_c_low_row,  g_macro_c_low_val,  current_idx)

                # Micro Trendlines (Green)
                g_micro_block_1 = green_candles.iloc[-5:]
                g_micro_block_2 = green_candles.iloc[-10:-5]

                g_micro_c_high_idx = g_micro_block_1['High'].iloc[::-1].idxmax()
                g_micro_c_low_idx  = g_micro_block_1['Low'].iloc[::-1].idxmin()
                g_micro_p_high_idx = g_micro_block_2['High'].iloc[::-1].idxmax()
                g_micro_p_low_idx  = g_micro_block_2['Low'].iloc[::-1].idxmin()

                g_mi_p_high_row = get_df_idx(g_micro_p_high_idx)
                g_mi_c_high_row = get_df_idx(g_micro_c_high_idx)
                g_mi_p_low_row  = get_df_idx(g_micro_p_low_idx)
                g_mi_c_low_row  = get_df_idx(g_micro_c_low_idx)

                g_micro_p_high_val = g_micro_block_2.loc[g_micro_p_high_idx, 'High']
                g_micro_c_high_val = g_micro_block_1.loc[g_micro_c_high_idx, 'High']
                g_micro_p_low_val  = g_micro_block_2.loc[g_micro_p_low_idx, 'Low']
                g_micro_c_low_val  = g_micro_block_1.loc[g_micro_c_low_idx, 'Low']

                g_micro_rp, _ = self.get_line_projection(g_mi_p_high_row, g_micro_p_high_val, g_mi_c_high_row, g_micro_c_high_val, current_idx)
                g_micro_sp, _ = self.get_line_projection(g_mi_p_low_row,  g_micro_p_low_val,  g_mi_c_low_row,  g_micro_c_low_val,  current_idx)

                # -----------------------------------------------------------
                # 2. RED CANDLE LOGIC (MACRO & MICRO)
                # -----------------------------------------------------------
                # Macro Channels (Red)
                r_macro_block_1 = red_candles.iloc[-15:]
                r_macro_block_2 = red_candles.iloc[-30:-15]

                r_macro_c_high_idx = r_macro_block_1['High'].iloc[::-1].idxmax()
                r_macro_c_low_idx  = r_macro_block_1['Low'].iloc[::-1].idxmin()
                r_macro_p_high_idx = r_macro_block_2['High'].iloc[::-1].idxmax()
                r_macro_p_low_idx  = r_macro_block_2['Low'].iloc[::-1].idxmin()

                r_m_p_high_row = get_df_idx(r_macro_p_high_idx)
                r_m_c_high_row = get_df_idx(r_macro_c_high_idx)
                r_m_p_low_row  = get_df_idx(r_macro_p_low_idx)
                r_m_c_low_row  = get_df_idx(r_macro_c_low_idx)

                r_macro_p_high_val = r_macro_block_2.loc[r_macro_p_high_idx, 'High']
                r_macro_c_high_val = r_macro_block_1.loc[r_macro_c_high_idx, 'High']
                r_macro_p_low_val  = r_macro_block_2.loc[r_macro_p_low_idx, 'Low']
                r_macro_c_low_val  = r_macro_block_1.loc[r_macro_c_low_idx, 'Low']

                r_macro_rp, _ = self.get_line_projection(r_m_p_high_row, r_macro_p_high_val, r_m_c_high_row, r_macro_c_high_val, current_idx)
                r_macro_sp, _ = self.get_line_projection(r_m_p_low_row,  r_macro_p_low_val,  r_m_c_low_row,  r_macro_c_low_val,  current_idx)

                # Micro Trendlines (Red)
                r_micro_block_1 = red_candles.iloc[-5:]
                r_micro_block_2 = red_candles.iloc[-10:-5]

                r_micro_c_high_idx = r_micro_block_1['High'].iloc[::-1].idxmax()
                r_micro_c_low_idx  = r_micro_block_1['Low'].iloc[::-1].idxmin()
                r_micro_p_high_idx = r_micro_block_2['High'].iloc[::-1].idxmax()
                r_micro_p_low_idx  = r_micro_block_2['Low'].iloc[::-1].idxmin()

                r_mi_p_high_row = get_df_idx(r_micro_p_high_idx)
                r_mi_c_high_row = get_df_idx(r_micro_c_high_idx)
                r_mi_p_low_row  = get_df_idx(r_micro_p_low_idx)
                r_mi_c_low_row  = get_df_idx(r_micro_c_low_idx)

                r_micro_p_high_val = r_micro_block_2.loc[r_micro_p_high_idx, 'High']
                r_micro_c_high_val = r_micro_block_1.loc[r_micro_c_high_idx, 'High']
                r_micro_p_low_val  = r_micro_block_2.loc[r_micro_p_low_idx, 'Low']
                r_micro_c_low_val  = r_micro_block_1.loc[r_micro_c_low_idx, 'Low']

                r_micro_rp, _ = self.get_line_projection(r_mi_p_high_row, r_micro_p_high_val, r_mi_c_high_row, r_micro_c_high_val, current_idx)
                r_micro_sp, _ = self.get_line_projection(r_mi_p_low_row,  r_micro_p_low_val,  r_mi_c_low_row,  r_micro_c_low_val,  current_idx)

                # -----------------------------------------------------------
                # TERMINAL LOGGING
                # -----------------------------------------------------------
                india_time = datetime.now(self.ist)
                print("=" * 70)
                if latest_green is not None:
                    print(f"LATEST GREEN | Time: {latest_green.name.strftime('%H:%M:%S')} | HIGH: {latest_green['High']} | Close: {latest_green['Close']}")
                if latest_red is not None:
                    print(f"LATEST RED   | Time: {latest_red.name.strftime('%H:%M:%S')} | LOW: {latest_red['Low']} | Close: {latest_red['Close']}")
                print("=" * 70)

                # Green Channel / Trendline Indexes and Limits
                print("[GREEN] macro resistance:", g_macro_p_high_idx, g_macro_p_high_val, '\t', g_macro_c_high_idx, g_macro_c_high_val)
                print("[GREEN] macro support:", g_macro_p_low_idx, g_macro_p_low_val, '\t', g_macro_c_low_idx, g_macro_c_low_val, '\n')
                print("[GREEN] micro resistance:", g_micro_p_high_idx, g_micro_p_high_val, '\t', g_micro_c_high_idx, g_micro_c_high_val)
                print("[GREEN] micro support:", g_micro_p_low_idx, g_micro_p_low_val, '\t', g_micro_c_low_idx, g_micro_c_low_val)
                print("-" * 60)

                # Red Channel / Trendline Indexes and Limits
                print("[RED] macro resistance:", r_macro_p_high_idx, r_macro_p_high_val, '\t', r_macro_c_high_idx, r_macro_c_high_val)
                print("[RED] macro support:", r_macro_p_low_idx, r_macro_p_low_val, '\t', r_macro_c_low_idx, r_macro_c_low_val, '\n')
                print("[RED] micro resistance:", r_micro_p_high_idx, r_micro_p_high_val, '\t', r_micro_c_high_idx, r_micro_c_high_val)
                print("[RED] micro support:", r_micro_p_low_idx, r_micro_p_low_val, '\t', r_micro_c_low_idx, r_micro_c_low_val)
                print("=" * 60)

                print(f"[{india_time.strftime('%H:%M:%S')}] Current Close: {current_close}")
                print(f"GREEN STRATEGY -> Macro [S:{g_macro_sp:.1f} R:{g_macro_rp:.1f}] | Micro [S:{g_micro_sp:.1f} R:{g_micro_rp:.1f}]")
                print(f"RED STRATEGY   -> Macro [S:{r_macro_sp:.1f} R:{r_macro_rp:.1f}] | Micro [S:{r_micro_sp:.1f} R:{r_micro_rp:.1f}]")
                print("-" * 70)

                time.sleep(10)

            except Exception as e:
                print(f"Error in runtime logic: {e}")
                time.sleep(10)

if __name__ == "__main__":
    API_KEY = "cQtdXuXL"
    CLIENT_CODE = "AABZ498970"
    PASSWORD = '1107'
    TOTP_TOKEN = "CGAODQ6O3SGLIBQWUSWFABIL2Y"
    bot = AngelLivePaperTrader(client_id=CLIENT_CODE, pwd=PASSWORD, totp=TOTP_TOKEN, access_token=API_KEY)
    bot.getting_logic()

[I 260724 06:46:37 smartConnect:121] in pool


LATEST GREEN | Time: 12:15:00 | HIGH: 23708.05 | Close: 23706.5
LATEST RED   | Time: 12:14:00 | LOW: 23701.85 | Close: 23702.85
[GREEN] macro resistance: 2026-07-24 11:45:00+05:30 23687.5 	 2026-07-24 12:05:00+05:30 23716.2
[GREEN] macro support: 2026-07-24 11:19:00+05:30 23645.95 	 2026-07-24 11:46:00+05:30 23685.3 

[GREEN] micro resistance: 2026-07-24 11:53:00+05:30 23710.7 	 2026-07-24 12:05:00+05:30 23716.2
[GREEN] micro support: 2026-07-24 12:01:00+05:30 23690.4 	 2026-07-24 12:04:00+05:30 23698.05
------------------------------------------------------------
[RED] macro resistance: 2026-07-24 11:44:00+05:30 23687.5 	 2026-07-24 12:06:00+05:30 23717.9
[RED] macro support: 2026-07-24 11:12:00+05:30 23617.1 	 2026-07-24 12:00:00+05:30 23689.25 

[RED] micro resistance: 2026-07-24 12:06:00+05:30 23717.9 	 2026-07-24 12:10:00+05:30 23715.2
[RED] micro support: 2026-07-24 12:00:00+05:30 23689.25 	 2026-07-24 12:14:00+05:30 23701.85
[12:16:55] Current Close: 23706.5
GREEN STRATEGY -> Ma

[E 260724 07:02:29 smartConnect:243] Error occurred while making a POST request to https://apiconnect.angelbroking.com/rest/secure/angelbroking/historical/v1/getCandleData. Error: Too many requests. URL: https://apiconnect.angelbroking.com/rest/secure/angelbroking/historical/v1/getCandleData, Headers: {'Content-type': 'application/json', 'X-ClientLocalIP': '127.0.0.1', 'X-ClientPublicIP': '106.193.147.98', 'X-MACAddress': '02:42:ac:1c:00:0c', 'Accept': 'application/json', 'X-PrivateKey': 'cQtdXuXL', 'X-UserType': 'USER', 'X-SourceID': 'WEB'}, Request: {'exchange': 'NSE', 'symboltoken': '99926000', 'interval': 'ONE_MINUTE', 'fromdate': '2026-07-09 12:31', 'todate': '2026-07-24 12:31'}, Response: {'message': 'Too many requests', 'errorcode': 'AB1021', 'status': False, 'data': None}


LATEST GREEN | Time: 12:31:00 | HIGH: 23794.65 | Close: 23791.4
LATEST RED   | Time: 12:30:00 | LOW: 23778.7 | Close: 23781.8
[GREEN] macro resistance: 2026-07-24 12:05:00+05:30 23716.2 	 2026-07-24 12:31:00+05:30 23794.65
[GREEN] macro support: 2026-07-24 11:42:00+05:30 23670.1 	 2026-07-24 12:15:00+05:30 23701.35 

[GREEN] micro resistance: 2026-07-24 12:24:00+05:30 23739.85 	 2026-07-24 12:31:00+05:30 23794.65
[GREEN] micro support: 2026-07-24 12:20:00+05:30 23713.75 	 2026-07-24 12:25:00+05:30 23736.9
------------------------------------------------------------
[RED] macro resistance: 2026-07-24 11:54:00+05:30 23710.05 	 2026-07-24 12:29:00+05:30 23792.3
[RED] macro support: 2026-07-24 11:32:00+05:30 23649.15 	 2026-07-24 12:00:00+05:30 23689.25 

[RED] micro resistance: 2026-07-24 12:07:00+05:30 23716.35 	 2026-07-24 12:29:00+05:30 23792.3
[RED] micro support: 2026-07-24 12:13:00+05:30 23704.8 	 2026-07-24 12:14:00+05:30 23701.85
[12:32:45] Current Close: 23791.4
GREEN STRATEGY ->


KeyboardInterrupt



In [ ]:
import pandas as pd
from SmartApi import SmartConnect
import pyotp
import pytz
from datetime import datetime, timedelta
import time

class AngelLivePaperTrader:
    def __init__(self, client_id, pwd, totp, access_token):
        self.current_totp = pyotp.TOTP(totp).now()
        self.obj = SmartConnect(api_key=access_token)
        self.data = self.obj.generateSession(client_id, pwd, self.current_totp)
        self.exchange = "NSE"
        self.symboltoken = "99926000"
        self.interval = "ONE_MINUTE"
        self.ist = pytz.timezone('Asia/Kolkata')

        # Trackers required by the pipeline
        self.formed_resistance_rays = []
        self.active_resistance_breakouts = []
        self.formed_support_rays = []
        self.active_support_breakdowns = []

    def getting_data(self):
        now_time = datetime.now(self.ist) - timedelta(minutes=1)
        end_date = now_time.strftime("%Y-%m-%d %H:%M")
        start_date = (now_time - pd.Timedelta(days=30)).strftime("%Y-%m-%d %H:%M")

        if not self.data or not self.data.get("status"):
            return None

        try:
            historic_params = {
                "exchange": self.exchange,
                "symboltoken": self.symboltoken,
                "interval": self.interval,
                "fromdate": start_date,
                "todate": end_date
            }
            raw_candles = self.obj.getCandleData(historic_params)
            if raw_candles and raw_candles.get("status") and raw_candles.get("data"):
                columns = ["Timestamp", "Open", "High", "Low", "Close", "Volume"]
                dataframe = pd.DataFrame(raw_candles["data"], columns=columns)
                return dataframe
            return None
        except Exception:
            return None

    def resample_dataframe(self, df_1m, timeframe="5min"):
        """
        Resamples 1-minute OHLCV data into target timeframe (e.g. '5min', '15min').
        """
        ohlc_dict = {
            'Open': 'first',
            'High': 'max',
            'Low': 'min',
            'Close': 'last',
            'Volume': 'sum'
        }
        resampled_df = df_1m.resample(timeframe).agg(ohlc_dict).dropna()
        return resampled_df

    def get_line_projection(self, start_row, start_val, end_row, end_val, current_row):
        if end_row == start_row:
            return end_val, 0.0
        slope = (end_val - start_val) / (end_row - start_row)
        projected_val = end_val + slope * (current_row - end_row)
        return projected_val, slope

    def separate_candles(self, df):
        green_candles = df[df['Close'] > df['Open']]
        red_candles = df[df['Close'] < df['Open']]

        latest_green = green_candles.iloc[-1] if not green_candles.empty else None
        latest_red = red_candles.iloc[-1] if not red_candles.empty else None

        return latest_green, latest_red, green_candles, red_candles

    def compute_trendline_levels(self, df, tf_label="1M"):
        """
        Calculates Macro/Micro green and red level projections for a given dataframe.
        """
        if df is None or df.empty or len(df) < 140:
            print(f"[{tf_label}] Insufficient data points (needs at least 140 candles).")
            return None

        current_idx = len(df) - 1
        current_close = df['Close'].iloc[-1]

        latest_green, latest_red, green_candles, red_candles = self.separate_candles(df)

        if len(green_candles) < 30 or len(red_candles) < 30:
            print(f"[{tf_label}] Not enough specific color candles to compute levels yet.")
            return None

        def get_df_idx(timestamp):
            return df.index.get_loc(timestamp)

        # -----------------------------------------------------------
        # 1. GREEN CANDLE LOGIC (MACRO & MICRO)
        # -----------------------------------------------------------
        g_macro_block_1 = green_candles.iloc[-15:]
        g_macro_block_2 = green_candles.iloc[-30:-15]

        g_macro_c_high_idx = g_macro_block_1['High'].iloc[::-1].idxmax()
        g_macro_c_low_idx  = g_macro_block_1['Low'].iloc[::-1].idxmin()
        g_macro_p_high_idx = g_macro_block_2['High'].iloc[::-1].idxmax()
        g_macro_p_low_idx  = g_macro_block_2['Low'].iloc[::-1].idxmin()

        g_m_p_high_row = get_df_idx(g_macro_p_high_idx)
        g_m_c_high_row = get_df_idx(g_macro_c_high_idx)
        g_m_p_low_row  = get_df_idx(g_macro_p_low_idx)
        g_m_c_low_row  = get_df_idx(g_macro_c_low_idx)

        g_macro_p_high_val = g_macro_block_2.loc[g_macro_p_high_idx, 'High']
        g_macro_c_high_val = g_macro_block_1.loc[g_macro_c_high_idx, 'High']
        g_macro_p_low_val  = g_macro_block_2.loc[g_macro_p_low_idx, 'Low']
        g_macro_c_low_val  = g_macro_block_1.loc[g_macro_c_low_idx, 'Low']

        g_macro_rp, _ = self.get_line_projection(g_m_p_high_row, g_macro_p_high_val, g_m_c_high_row, g_macro_c_high_val, current_idx)
        g_macro_sp, _ = self.get_line_projection(g_m_p_low_row,  g_macro_p_low_val,  g_m_c_low_row,  g_macro_c_low_val,  current_idx)

        g_micro_block_1 = green_candles.iloc[-5:]
        g_micro_block_2 = green_candles.iloc[-10:-5]

        g_micro_c_high_idx = g_micro_block_1['High'].iloc[::-1].idxmax()
        g_micro_c_low_idx  = g_micro_block_1['Low'].iloc[::-1].idxmin()
        g_micro_p_high_idx = g_micro_block_2['High'].iloc[::-1].idxmax()
        g_micro_p_low_idx  = g_micro_block_2['Low'].iloc[::-1].idxmin()

        g_mi_p_high_row = get_df_idx(g_micro_p_high_idx)
        g_mi_c_high_row = get_df_idx(g_micro_c_high_idx)
        g_mi_p_low_row  = get_df_idx(g_micro_p_low_idx)
        g_mi_c_low_row  = get_df_idx(g_micro_c_low_idx)

        g_micro_p_high_val = g_micro_block_2.loc[g_micro_p_high_idx, 'High']
        g_micro_c_high_val = g_micro_block_1.loc[g_micro_c_high_idx, 'High']
        g_micro_p_low_val  = g_micro_block_2.loc[g_micro_p_low_idx, 'Low']
        g_micro_c_low_val  = g_micro_block_1.loc[g_micro_c_low_idx, 'Low']

        g_micro_rp, _ = self.get_line_projection(g_mi_p_high_row, g_micro_p_high_val, g_mi_c_high_row, g_micro_c_high_val, current_idx)
        g_micro_sp, _ = self.get_line_projection(g_mi_p_low_row,  g_micro_p_low_val,  g_mi_c_low_row,  g_micro_c_low_val,  current_idx)

        # -----------------------------------------------------------
        # 2. RED CANDLE LOGIC (MACRO & MICRO)
        # -----------------------------------------------------------
        r_macro_block_1 = red_candles.iloc[-15:]
        r_macro_block_2 = red_candles.iloc[-30:-15]

        r_macro_c_high_idx = r_macro_block_1['High'].iloc[::-1].idxmax()
        r_macro_c_low_idx  = r_macro_block_1['Low'].iloc[::-1].idxmin()
        r_macro_p_high_idx = r_macro_block_2['High'].iloc[::-1].idxmax()
        r_macro_p_low_idx  = r_macro_block_2['Low'].iloc[::-1].idxmin()

        r_m_p_high_row = get_df_idx(r_macro_p_high_idx)
        r_m_c_high_row = get_df_idx(r_macro_c_high_idx)
        r_m_p_low_row  = get_df_idx(r_macro_p_low_idx)
        r_m_c_low_row  = get_df_idx(r_macro_c_low_idx)

        r_macro_p_high_val = r_macro_block_2.loc[r_macro_p_high_idx, 'High']
        r_macro_c_high_val = r_macro_block_1.loc[r_macro_c_high_idx, 'High']
        r_macro_p_low_val  = r_macro_block_2.loc[r_macro_p_low_idx, 'Low']
        r_macro_c_low_val  = r_macro_block_1.loc[r_macro_c_low_idx, 'Low']

        r_macro_rp, _ = self.get_line_projection(r_m_p_high_row, r_macro_p_high_val, r_m_c_high_row, r_macro_c_high_val, current_idx)
        r_macro_sp, _ = self.get_line_projection(r_m_p_low_row,  r_macro_p_low_val,  r_m_c_low_row,  r_macro_c_low_val,  current_idx)

        r_micro_block_1 = red_candles.iloc[-5:]
        r_micro_block_2 = red_candles.iloc[-10:-5]

        r_micro_c_high_idx = r_micro_block_1['High'].iloc[::-1].idxmax()
        r_micro_c_low_idx  = r_micro_block_1['Low'].iloc[::-1].idxmin()
        r_micro_p_high_idx = r_micro_block_2['High'].iloc[::-1].idxmax()
        r_micro_p_low_idx  = r_micro_block_2['Low'].iloc[::-1].idxmin()

        r_mi_p_high_row = get_df_idx(r_micro_p_high_idx)
        r_mi_c_high_row = get_df_idx(r_micro_c_high_idx)
        r_mi_p_low_row  = get_df_idx(r_micro_p_low_idx)
        r_mi_c_low_row  = get_df_idx(r_micro_c_low_idx)

        r_micro_p_high_val = r_micro_block_2.loc[r_micro_p_high_idx, 'High']
        r_micro_c_high_val = r_micro_block_1.loc[r_micro_c_high_idx, 'High']
        r_micro_p_low_val  = r_micro_block_2.loc[r_micro_p_low_idx, 'Low']
        r_micro_c_low_val  = r_micro_block_1.loc[r_micro_c_low_idx, 'Low']

        r_micro_rp, _ = self.get_line_projection(r_mi_p_high_row, r_micro_p_high_val, r_mi_c_high_row, r_micro_c_high_val, current_idx)
        r_micro_sp, _ = self.get_line_projection(r_mi_p_low_row,  r_micro_p_low_val,  r_mi_c_low_row,  r_micro_c_low_val,  current_idx)

        # -----------------------------------------------------------
        # LOGGING OUTPUT FOR SPECIFIC TIMEFRAME
        # -----------------------------------------------------------
        print("=" * 70)
        print(f"                       TIMEFRAME: {tf_label}")
        print("=" * 70)
        if latest_green is not None:
            print(f"LATEST GREEN | Time: {latest_green.name.strftime('%H:%M:%S')} | HIGH: {latest_green['High']} | Close: {latest_green['Close']}")
        if latest_red is not None:
            print(f"LATEST RED   | Time: {latest_red.name.strftime('%H:%M:%S')} | LOW: {latest_red['Low']} | Close: {latest_red['Close']}")
        print("=" * 70)

        print(f"[{tf_label} GREEN] macro resistance:", g_macro_p_high_idx, g_macro_p_high_val, '\t', g_macro_c_high_idx, g_macro_c_high_val)
        print(f"[{tf_label} GREEN] macro support:", g_macro_p_low_idx, g_macro_p_low_val, '\t', g_macro_c_low_idx, g_macro_c_low_val, '\n')
        print(f"[{tf_label} GREEN] micro resistance:", g_micro_p_high_idx, g_micro_p_high_val, '\t', g_micro_c_high_idx, g_micro_c_high_val)
        print(f"[{tf_label} GREEN] micro support:", g_micro_p_low_idx, g_micro_p_low_val, '\t', g_micro_c_low_idx, g_micro_c_low_val)
        print("-" * 60)

        print(f"[{tf_label} RED] macro resistance:", r_macro_p_high_idx, r_macro_p_high_val, '\t', r_macro_c_high_idx, r_macro_c_high_val)
        print(f"[{tf_label} RED] macro support:", r_macro_p_low_idx, r_macro_p_low_val, '\t', r_macro_c_low_idx, r_macro_c_low_val, '\n')
        print(f"[{tf_label} RED] micro resistance:", r_micro_p_high_idx, r_micro_p_high_val, '\t', r_micro_c_high_idx, r_micro_c_high_val)
        print(f"[{tf_label} RED] micro support:", r_micro_p_low_idx, r_micro_p_low_val, '\t', r_micro_c_low_idx, r_micro_c_low_val)
        print("=" * 60)

        print(f"[{tf_label}] Current Close: {current_close}")
        print(f"[{tf_label}] GREEN STRATEGY -> Macro [S:{g_macro_sp:.1f} R:{g_macro_rp:.1f}] | Micro [S:{g_micro_sp:.1f} R:{g_micro_rp:.1f}]")
        print(f"[{tf_label}] RED STRATEGY   -> Macro [S:{r_macro_sp:.1f} R:{r_macro_rp:.1f}] | Micro [S:{r_micro_sp:.1f} R:{r_micro_rp:.1f}]")
        print("-" * 70)

    def getting_logic(self):
        while True:
            try:
                df_1m = self.getting_data()
                if df_1m is None or df_1m.empty or len(df_1m) < 140:
                    time.sleep(10)
                    continue

                # Clean and prepare 1-minute dataframe
                df_1m['Timestamp'] = pd.to_datetime(df_1m['Timestamp'])
                df_1m.set_index('Timestamp', inplace=True)
                df_1m.index = df_1m.index.tz_convert(self.ist)
                df_1m = df_1m.dropna(subset=['Open', 'High', 'Low', 'Close'])

                # Resample into 5m and 15m timeframes
                df_5m = self.resample_dataframe(df_1m, timeframe="5min")
                df_15m = self.resample_dataframe(df_1m, timeframe="15min")

                india_time = datetime.now(self.ist)
                print(f"\n==================== RUN TIME: {india_time.strftime('%Y-%m-%d %H:%M:%S')} ====================")

                # Compute and print logic across timeframes
                self.compute_trendline_levels(df_1m, tf_label="1-MINUTE")
                self.compute_trendline_levels(df_5m, tf_label="5-MINUTE")
                self.compute_trendline_levels(df_15m, tf_label="15-MINUTE")

                time.sleep(60)

            except Exception as e:
                print(f"Error in runtime logic: {e}")
                time.sleep(10)

if __name__ == "__main__":
    API_KEY = "cQtdXuXL"
    CLIENT_CODE = "AABZ498970"
    PASSWORD = '1107'
    TOTP_TOKEN = "CGAODQ6O3SGLIBQWUSWFABIL2Y"
    bot = AngelLivePaperTrader(client_id=CLIENT_CODE, pwd=PASSWORD, totp=TOTP_TOKEN, access_token=API_KEY)
    bot.getting_logic()

[I 260724 16:49:04 smartConnect:121] in pool



==================== RUN TIME: 2026-07-24 22:19:06 ====================
                       TIMEFRAME: 1-MINUTE
LATEST GREEN | Time: 15:29:00 | HIGH: 23792.95 | Close: 23787.0
LATEST RED   | Time: 15:25:00 | LOW: 23773.4 | Close: 23776.4
[1-MINUTE GREEN] macro resistance: 2026-07-24 14:53:00+05:30 23793.0 	 2026-07-24 15:29:00+05:30 23792.95
[1-MINUTE GREEN] macro support: 2026-07-24 15:02:00+05:30 23747.65 	 2026-07-24 15:09:00+05:30 23751.35 

[1-MINUTE GREEN] micro resistance: 2026-07-24 15:15:00+05:30 23789.8 	 2026-07-24 15:29:00+05:30 23792.95
[1-MINUTE GREEN] micro support: 2026-07-24 15:18:00+05:30 23767.45 	 2026-07-24 15:26:00+05:30 23772.05
------------------------------------------------------------
[1-MINUTE RED] macro resistance: 2026-07-24 14:54:00+05:30 23794.15 	 2026-07-24 15:16:00+05:30 23789.5
[1-MINUTE RED] macro support: 2026-07-24 14:48:00+05:30 23761.85 	 2026-07-24 15:01:00+05:30 23750.1 

[1-MINUTE RED] micro resistance: 2026-07-24 15:07:00+05:30 23764.25 

KeyboardInterrupt: 